# Dates and Times: Analysing Order and Delivery Dates

In this practice you will convert date strings into date objects, format and extract parts of dates, and calculate delivery times. You will then use pandas to create date-based columns for the whole dataset.

## Background

The operations team wants to know which months and weekdays are busiest, and how long deliveries take. The dates in the CSV file are stored as text, so they must be converted before Python can do any of this.

## Data Description

All the practices in this module use **Retail Orders.csv**: 200 orders from 2025 placed with a small online retailer based in Ireland. The data is fictional and was created for teaching.

| Column Name | Description |
|---|---|
| order_id | Unique order identifier |
| order_date | Date the order was placed (YYYY-MM-DD) |
| delivery_date | Date the order was delivered; blank if not yet delivered |
| customer_name | Customer's name, as typed into the order system |
| city | Delivery city |
| product | Product ordered |
| category | Product category: Electronics, Office or Home |
| quantity | Number of units ordered |
| unit_price | Price of one unit, in euro |
| priority | Order priority: Urgent, Normal or Minor |
| status | Order status: Delivered, Shipped or Cancelled |

## 1. Load the Dataset

We import pandas and read the CSV file into a DataFrame. Keep **Retail Orders.csv** in the same folder as this notebook.

In [1]:
import pandas as pd

df = pd.read_csv("Retail Orders.csv")
df.head()

,order_id,order_date,delivery_date,customer_name,city,product,category,quantity,unit_price,priority,status
0,ORD-1001,2025-01-11,2025-01-19,Tadhg Farrell,Limerick,Filing Cabinet,Office,5,120.0,Minor,Delivered
1,ORD-1002,2025-01-11,2025-01-15,Liam Kelly,Dublin,Whiteboard,Office,3,45.0,Normal,Delivered
2,ORD-1003,2025-01-11,2025-01-13,Sean Ryan,Cork,A4 Notebook,Office,3,4.5,Normal,Delivered
3,ORD-1004,2025-01-12,2025-01-16,Sinead Ryan,Cork,Ergonomic Chair,Office,5,189.0,Normal,Delivered
4,ORD-1005,2025-01-15,2025-01-24,Tomas Lynch,Galway,A4 Notebook,Office,1,4.5,Minor,Delivered


## 2. Convert a String to a Date

`order_date` is text. `datetime.strptime()` converts it into a datetime object, using a format string that describes how the date is written.

In [2]:
from datetime import datetime

order_text = df["order_date"][0]
print(order_text, type(order_text))

order_dt = datetime.strptime(order_text, "%Y-%m-%d")
print(order_dt, type(order_dt))

2025-01-11 <class 'str'>
2025-01-11 00:00:00 <class 'datetime.datetime'>


## 3. Format a Date with strftime()

`strftime()` turns a date back into text in any format we choose. `%d` is the day, `%B` the full month name, `%b` the short month name, `%A` the weekday and `%Y` the four-digit year.

In [3]:
print(order_dt.strftime("%d %B %Y"))
print(order_dt.strftime("%d/%b/%Y"))
print(order_dt.strftime("%A"))
print(order_dt.strftime("%Y%B"))

11 January 2025
11/Jan/2025
Saturday
2025January


## 4. Extract Parts of a Date

Attributes give each part as a number. `pd.Timestamp()` also gives the quarter.

In [4]:
import pandas as pd

print(order_dt.year, order_dt.month, order_dt.day)
print(pd.Timestamp(order_dt).quarter)

2025 1 11
1


## 5. Today's Date and the Current Time

`date.today()` and `datetime.now()` read your computer's clock, so your output will differ from the one shown.

In [5]:
from datetime import date

print(date.today())
now = datetime.now()
print(now.hour, now.minute, now.second)

2026-10-19
12 5 27


## 6. Calculate a Delivery Time

Subtracting two dates gives a `timedelta`. Its `.days` attribute is the number of days between them.

In [6]:
delivery_dt = datetime.strptime(df["delivery_date"][0], "%Y-%m-%d")
gap = delivery_dt - order_dt
print(gap)
print(gap.days, "days")

8 days, 0:00:00
8 days


Adding a `timedelta` moves a date forward. The retailer accepts returns for 30 days after delivery:

In [7]:
from datetime import timedelta

returns_deadline = delivery_dt + timedelta(days=30)
print(returns_deadline.strftime("%d %B %Y"))

18 February 2025


## 7. Convert Whole Columns with pd.to_datetime()

For a whole column, `pd.to_datetime()` is much quicker than converting one value at a time. Blank delivery dates become `NaT` (not a time).

In [8]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["delivery_date"] = pd.to_datetime(df["delivery_date"])
df[["order_id", "order_date", "delivery_date", "status"]].head(8)

,order_id,order_date,delivery_date,status
0,ORD-1001,2025-01-11,2025-01-19,Delivered
1,ORD-1002,2025-01-11,2025-01-15,Delivered
2,ORD-1003,2025-01-11,2025-01-13,Delivered
3,ORD-1004,2025-01-12,2025-01-16,Delivered
4,ORD-1005,2025-01-15,2025-01-24,Delivered
5,ORD-1006,2025-01-22,NaT,Shipped
6,ORD-1007,2025-01-23,2025-02-01,Delivered
7,ORD-1008,2025-01-24,NaT,Shipped


## 8. Create Date Features with the .dt Accessor

`.dt` extracts parts of every date in a column at once. We add the month name, weekday name and quarter, and the delivery time in days.

Orders that have not been delivered have no delivery time. Pandas stores those gaps as `NaN`, which makes the column decimal, so delivery times show as `8.0` rather than `8`.

In [9]:
df["order_month"] = df["order_date"].dt.month_name()
df["order_weekday"] = df["order_date"].dt.day_name()
df["order_quarter"] = df["order_date"].dt.quarter
df["delivery_days"] = (df["delivery_date"] - df["order_date"]).dt.days

df[["order_id", "order_month", "order_weekday", "order_quarter", "delivery_days"]].head()

,order_id,order_month,order_weekday,order_quarter,delivery_days
0,ORD-1001,January,Saturday,1,8.0
1,ORD-1002,January,Saturday,1,4.0
2,ORD-1003,January,Saturday,1,2.0
3,ORD-1004,January,Sunday,1,4.0
4,ORD-1005,January,Wednesday,1,9.0


`value_counts()` counts how often each value appears, so we can see the busiest weekdays:

In [10]:
df["order_weekday"].value_counts()

order_weekday
Monday       39
Friday       33
Saturday     31
Thursday     29
Sunday       26
Tuesday      24
Wednesday    18
Name: count, dtype: int64

## 9. Merge Separate Date Columns

Some systems store a date as separate year, month and day columns. `pd.to_datetime()` can merge them back into a single date.

In [11]:
parts = pd.DataFrame({
    "order_id": ["ORD-1001", "ORD-1002", "ORD-1003"],
    "year": [2025, 2025, 2025],
    "month": [1, 1, 1],
    "day": [11, 11, 11],
})
parts["date"] = pd.to_datetime(parts[["year", "month", "day"]])
parts

,order_id,year,month,day,date
0,ORD-1001,2025,1,11,2025-01-11
1,ORD-1002,2025,1,11,2025-01-11
2,ORD-1003,2025,1,11,2025-01-11


## 10. Handle Inconsistent Date Formats

Dates typed by hand often use different formats. With `format="mixed"`, `pd.to_datetime()` works out each format, as long as the order of day, month and year is consistent.

In [12]:
typed = ["11jan25", "12 January 2025", "15jan2025", "22 Jan 2025"]
pd.to_datetime(typed, format="mixed")

DatetimeIndex(['2025-01-11', '2025-01-12', '2025-01-15', '2025-01-22'], dtype='datetime64[us]', freq=None)

## Summary

In this practice you:

- converted text to dates with `datetime.strptime()` and back to text with `strftime()`
- extracted the year, month, day and quarter from a date
- read today's date and the current time
- calculated delivery times with date subtraction and moved dates with `timedelta`
- converted whole columns with `pd.to_datetime()` and created date features with `.dt`
- merged separate year, month and day columns, and parsed inconsistently typed dates

In Lesson 10 you will use `lambda` and `map()` to transform whole lists of values in one line.